# Erkennung und Behandlung von Ausreißern

### Was sind Ausreißer?


> Wikipedia-Definition:
> In der Statistik ist ein Ausreißer ein Beobachtungspunkt, der von anderen Beobachtungen entfernt ist.

Ausreißer sind Extremwerte, die von anderen Beobachtungen in den Daten abweichen. Sie können auf eine Variabilität in einer Messung, experimentelle Fehler oder eine Neuheit hinweisen. Mit anderen Worten ist ein Ausreißer eine Beobachtung, die von einem allgemeinen Muster einer Stichprobe abweicht.

Das Data-Science-Projekt beginnt mit der Sammlung von Daten, und genau dann werden Ausreißer erstmals in die Population eingeführt. Sie werden jedoch in der Sammelphase überhaupt nichts über die Ausreißer wissen. Die Ausreißer können das Ergebnis eines Fehlers während der Datensammlung sein oder einfach nur ein Hinweis auf Varianz in Ihren Daten.

Die häufigsten Ursachen für Ausreißer in einem Datensatz:
- Dateneingabefehler (menschliche Fehler)
- Messfehler (Instrumentenfehler)
- Experimentelle Fehler (Datenextraktions- oder Experimentplanungs-/Ausführungsfehler)
- Absichtlich (Dummy-Ausreißer, die erstellt wurden, um Erkennungsmethoden zu testen)
- Datenverarbeitungsfehler (Datenmanipulation oder unbeabsichtigte Datensatzmutationen)
- Stichprobenfehler (Extraktion oder Vermischung von Daten aus falschen oder verschiedenen Quellen)
- Natürlich (kein Fehler, Neuheiten in den Daten)

Das Ziel der Ausreißererkennung ist es also nicht nur, Ausreißer zu finden, sondern auch ihren Ursprung zu finden. Dazu müssen wir den Datensatz und den Datensammlungsprozess verstehen.

Es gibt zwei Arten von Ausreißern: **univariate Ausreißer** und **multivariate Ausreißer**. Univariate Ausreißer können gefunden werden, wenn man sich eine Verteilung von Werten in einem einzelnen Merkmalsraum ansieht. Multivariate Ausreißer können in einem n-dimensionalen Raum (von n-Merkmalen) gefunden werden.


Im maschinellen Lernen und in jeder quantitativen Disziplin ist die Qualität der Daten genauso wichtig wie die Qualität eines Vorhersage- oder Klassifikationsmodells.

Wenn Sie eine Ausreißererkennungssuche starten, müssen Sie auch zwei wichtige Fragen zu Ihrem Datensatz beantworten:
- Welche und wie viele Merkmale berücksichtige ich zur Erkennung von Ausreißern? (**univariat** / **multivariat**)
- Kann ich eine Verteilung(en) von Werten für meine ausgewählten Merkmale annehmen? (**parametrisch** / **nicht-parametrisch**)

In diesem Notebook werden wir nur univariate parametrische Ausreißer betrachten. Aber wir werden später im Bootcamp lernen, wie man multivariate Ausreißer erkennt. Wir werden das Modul [`scipy.stats`](https://docs.scipy.org/doc/scipy/reference/stats.html) verwenden, um Ausreißer zu erkennen, indem wir den [`z-score`](https://en.wikipedia.org/wiki/Standard_score) jedes Merkmals berechnen.


### Ausreißer finden


#### Der visuelle Weg

Sie können Visualisierungen verwenden, um Ausreißer zu finden. Es gibt drei Arten von Plots, die besonders gut zur Visualisierung von Ausreißern geeignet sind:
- Boxplots
- Histogramme
- Streudiagramme

#### Der mathematische Weg

Die Intuition hinter dem Z-Score besteht darin, jeden Datenpunkt zu beschreiben, indem man seine Beziehung zur Standardabweichung und zum Mittelwert der Datengruppe findet. Der Z-Score findet die Verteilung der Daten, bei der der Mittelwert 0 und die Standardabweichung 1 ist, d.h. Normalverteilung.
Dies macht Z-Score zu einer parametrischen Methode. Sehr häufig werden Datenpunkte nicht durch eine Gaußsche Verteilung beschrieben, dieses Problem kann durch Anwendung von Transformationen auf die Daten gelöst werden, d.h. Skalierung.

Sie fragen sich vielleicht, wie dies bei der Identifizierung der Ausreißer hilft? Nun, während der Berechnung des Z-Scores skalieren und zentrieren wir die Daten neu und suchen nach Datenpunkten, die zu weit von null entfernt sind.

Einige Python-Bibliotheken wie scipy und scikit-learn haben einfach zu verwendende Funktionen und Klassen für eine einfache Implementierung zusammen mit pandas und NumPy.
Nach den entsprechenden Transformationen des ausgewählten Merkmalsraums des Datensatzes kann der Z-Score eines beliebigen Datenpunkts mit folgendem Ausdruck berechnet werden:

$$ z = \frac{x - \mu}{\sigma} $$

wobei:
- $x$ der Wert des Merkmals ist
- $\mu$ der Mittelwert des Merkmals ist
- $\sigma$ die Standardabweichung des Merkmals ist

Bei der Berechnung des Z-Scores für jede Stichprobe im Datensatz muss ein Schwellenwert angegeben werden. Einige gute Faustregeln für Schwellenwerte können sein: 2,5, 3, 3,5 oder mehr Standardabweichungen.

Durch "Markieren" oder Entfernen der Datenpunkte, die über einem bestimmten Schwellenwert liegen, klassifizieren wir Daten in Ausreißer und Nicht-Ausreißer.


Z-Score Vorteile:
- Es ist eine sehr effektive Methode, wenn Sie die Werte im Merkmalsraum mit einer Gaußschen Verteilung beschreiben können. (Parametrisch)
- Die Implementierung ist sehr einfach mit pandas und scipy.stats Bibliotheken.

Z-Score Nachteile:
- Es ist nur bequem in einem niedrigdimensionalen Merkmalsraum in einem kleinen bis mittelgroßen Datensatz zu verwenden.
- Wird nicht empfohlen, wenn Verteilungen nicht als parametrisch angenommen werden können.

## Import 



Lassen Sie uns also die Bibliotheken und den Datensatz importieren.

In [ ]:
# Import the Libraries
from scipy.stats import zscore, boxcox
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Libraries for modelling
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    root_mean_squared_error,  # available in scikit-learn >= 1.4
)

# Set the RSEED for the train-test-split
RSEED = 42

# Set global variables for plots
plt.style.use('fivethirtyeight')
sns.set_color_codes('bright')
plt.rcParams["figure.figsize"] = (15, 10)

Importieren des Wine-Datensatzes.

In [ ]:
# Load the data
df = pd.read_csv('data/winequality-white.csv', sep=';')
df.drop('quality', axis=1, inplace=True)
# rename the columns
df.columns = df.columns.str.replace(' ', '_')
df.head()

Sie müssen sicherstellen, dass Sie die Train-Test-Aufteilung vornehmen, bevor Sie nach Ausreißern suchen. Wir behandeln unseren Testsatz als unsichtbare Daten, um die Leistung unseres Modells am Ende bewerten zu können. Wenn wir auch Ausreißer im Testsatz entfernen würden, würde unser Modell höchstwahrscheinlich besser auf unserem Testdatensatz abschneiden als es am Ende mit **echten** neuen Daten der Fall sein wird.

In [ ]:
# Define features and target
y = df['alcohol']
X = df.drop('alcohol', axis=1)

In [ ]:
df.columns

In [ ]:
# Split data into train and test set
X_train, X_test, y_train, y_test = train_test_split(X, y,random_state=RSEED)

## Umgang mit Ausreißern



Schauen wir uns die Verteilung der Merkmale im Trainingssatz mit verschiedenen Plots an. Da wir unsere Zielvariable für die Streudiagramme benötigen, verketten wir sie mit unseren Merkmalen.

In [ ]:
# Join features and target for plotting
df_train = pd.concat([X_train, y_train], axis=1)

Beginnen wir mit einigen Histogrammen.

In [ ]:
# Plot histograms for train data
fig,ax = plt.subplots(4,3,figsize=(16,16))
count = 0
for item in X_train.columns.tolist():
    sns.histplot(X_train[item], kde=True, ax=ax[int(count/3)][count%3], color='#33658A').set(title=item, xlabel='')
    count += 1
ax.flat[-2].set_visible(False)
ax.flat[-1].set_visible(False)
fig.tight_layout(pad=3)

In [ ]:
# Plot the training data in box plots

Und schauen wir uns die Korrelation zwischen den Merkmalen an.

In [ ]:
# Plot correlation matrix 
mask = np.triu(df_train.corr())
plt.figure(figsize = (20,15))
ax = sns.heatmap(round(df_train.corr(), 1)
                 ,annot=True
                 ,mask=mask
                 ,cmap='RdBu_r')

Schauen wir uns einzelne Merkmale genauer an, beginnend mit Chloriden. Wenn Sie mehr Merkmale sehen möchten, können Sie natürlich Zellen hinzufügen und den Code für die anwenden, die Sie sich genauer ansehen möchten!

In [ ]:
# Plotting one feature with histogram, box plot and scatter plot
fig,ax = plt.subplots(nrows=4,figsize=(16,16))
count = 0
# Histogram
sns.histplot(X_train['chlorides'], kde=True, ax=ax[0], color='#33658A').set(title='Chlorides', xlabel='')
# Box plot
sns.boxplot(x= X_train['chlorides'],ax=ax[1], color='#33658A').set(title='Chlorides')
# Scatter plot
sns.scatterplot(x=X_train.chlorides, y=df_train.alcohol,ax=ax[2], color='#33658A').set(title='Chlorides')
   
ax.flat[-1].set_visible(False)
fig.tight_layout(pad=3)

Sie können sehen, dass es an beiden Enden der Verteilung Ausreißer zu geben scheint. Da die Verteilung jedoch eindeutig rechtsschief ist, gibt es mehr Ausreißer auf der rechten Seite.
Ein erster Schritt, wenn Sie auf Ausreißer stoßen, besteht darin, die Daten zu transformieren. Manchmal hilft die Transformation der Daten bereits dabei, sie loszuwerden. Es gibt mehrere Möglichkeiten, die Daten zu transformieren, hier sind einige Beispiele:
- Log-Transformation
- Quadratwurzel-Transformation
- Potenz-Transformation
- Box-Cox-Transformation

Lassen Sie uns die Log-Transformation für unsere Daten ausprobieren. Wie Sie auf den obigen Histogrammen sehen können, haben wir mehrere Merkmale mit schiefen Daten. Wir werden die Transformation auf alle anwenden.

In [ ]:
# Log-transform skewed features using np.log()
X_train.volatile_acidity = X_train.volatile_acidity.apply(lambda x: np.log(x))
X_train.chlorides = X_train.chlorides.apply(lambda x: np.log(x))
X_train.density = X_train.density.apply(lambda x: np.log(x))
X_train.pH = X_train.pH.apply(lambda x: np.log(x))

Jetzt können wir überprüfen, ob die Transformation uns geholfen hat, die Ausreißer loszuwerden. Lassen Sie uns die Chlorid-Spalte erneut plotten.

In [ ]:
# Plotting one feature with histogram, box plot and scatter plot
fig,ax = plt.subplots(nrows=4,figsize=(16,16))
count = 0
# Histogram
sns.histplot(X_train['chlorides'], kde=True,ax=ax[0], color='#33658A').set(title='Chlorides', xlabel='')
# Box plot
sns.boxplot(x=X_train['chlorides'],ax=ax[1], color='#33658A').set(title='Chlorides')
# Scatter plot
sns.scatterplot(x=X_train.chlorides, y=df_train.alcohol,ax=ax[2], color='#33658A').set(title='Chlorides')
   
ax.flat[-1].set_visible(False)
fig.tight_layout(pad=3)

Wir haben die Schiefe losgeworden, aber es gibt immer noch Ausreißer. Lassen Sie uns also den Z-Score für die Merkmale nach der Transformation berechnen. `Scipy` bietet uns eine einfache Möglichkeit, den Z-Score für unseren gesamten Datensatz zu berechnen:

In [ ]:
# Calculate z-score for whole dataset
df_zscore  = zscore(df_train)
df_zscore

Wenn wir erwägen, Ausreißer basierend auf dem Z-Score oder einem anderen Schwellenwert zu entfernen, müssen wir berechnen, welchen Prozentsatz der Daten wir verlieren würden. Wenn der Anteil, den wir entfernen würden, zu groß ist, müssen wir unseren Ansatz überdenken. 

Berechnen wir also, wie viele Zeilen wir löschen würden, wenn wir die Ausreißer mit einem Schwellenwert von 3 Standardabweichungen entfernen würden.

In [ ]:
# Calculate data loss based on 3 standard deviations
num_outlier = len(df_train[(np.abs(zscore(df_train)) > 3).any(axis=1)])
print(f'{num_outlier} rows of {len(df_train)} total rows contain outliers.')
print(f'Percentage of data loss: {num_outlier/len(df_train)*100:.2f}%')

Ungefähr 8% der Beobachtungen in unserem Dataframe enthalten einen oder mehrere Ausreißer. 8% ist ziemlich viel, besonders wenn Sie einen kleinen Datensatz haben. Dennoch werden wir fortfahren, einen Dataframe ohne diese Beobachtungen zu erstellen. Daher können wir den Z-Score für jede Spalte berechnen und den Absolutwert nehmen, um unseren Dataframe abzufragen und nur die Zeilen mit Z-Scores unter 3 zu behalten.

In [ ]:
# Calculate z-score and query dataframe to remove rows with z-scores > 3
X_train_wo_outlier = X_train[(np.abs(zscore(df_train.select_dtypes(exclude='object'))) < 3).all(axis=1)]

Auf diese Weise haben wir die Zeilen in unserem Merkmals-Dataframe losgeworden. Aber jetzt müssen wir sicherstellen, dass wir auch die entsprechenden Einträge in unserer Zielvariable löschen.
Um dieselben Zeilen in y_train zu löschen, müssen wir df_train_wo_outlier und y_train zusammenführen und sie anschließend wieder aufteilen.

In [ ]:
# Remove outliers in target variable 
df_train_wo_outlier = pd.merge(X_train_wo_outlier, y_train, left_index=True, right_index=True)
y_train_wo_outlier = df_train_wo_outlier['alcohol']
X_train_wo_outlier = df_train_wo_outlier.drop('alcohol', axis=1)

Das hat funktioniert! Wir haben die Ausreißer aus unserem Dataframe entfernt. Schauen wir uns jetzt an, wie die Verteilung aussieht.

In [ ]:
# Plot histograms for cleaned features
fig,ax = plt.subplots(4,3,figsize=(16,16))
count = 0
for item in X_train_wo_outlier.columns.tolist():
    sns.histplot(X_train_wo_outlier[item], kde=True,ax=ax[int(count/3)][count%3], color='#33658A').set(title=item, xlabel='')
    count += 1
ax.flat[-1].set_visible(False)
ax.flat[-2].set_visible(False)
fig.tight_layout(pad=3)

In [ ]:
# And again you can plot the box plots using the code from above

Hat sich auch die Korrelationsmatrix verändert? Schauen wir uns die Korrelationsmatrix an.

In [ ]:
# Plot correlation matrix for cleaned data
mask = np.triu(X_train_wo_outlier.corr())
plt.figure(figsize = (20,15))
ax = sns.heatmap(round(X_train_wo_outlier.corr(),1)
                 ,annot=True
                 ,mask=mask
                 ,cmap='RdBu_r')

## Einfluss der Ausreißerentfernung auf die Modellleistung



Vergleichen wir nun zwei Modelle. Das erste ist ein einfaches lineares Regressionsmodell, das auf den Daten ohne Entfernung von Ausreißern trainiert wurde. Das zweite ist ein lineares Regressionsmodell, das auf den Daten mit Entfernung von Ausreißern trainiert wurde. Bevor wir den Testsatz zur Evaluierung unserer Modelle verwenden können, müssen wir ihn auf die gleiche Weise transformieren wie den Trainingssatz.

In [ ]:
# To-Do: Apply log transformation from train set to test set as well

# Your Code!

### Modell mit Ausreißern

Für das Modell, das auf den Daten mit Ausreißern trainiert wird, verwenden wir `X_train` und `y_train`.

In [ ]:
# Instantiate our model 
lin_reg = LinearRegression()

# Fit the model using our train data 
lin_reg.fit(X_train, y_train)

# Make predictions on the train data  and on the test data
y_pred_train  = lin_reg.predict(X_train)
y_pred = lin_reg.predict(X_test)

### Modell ohne Ausreißer

Dieses Modell wird auf unseren bereinigten Daten `X_train_wo_outlier` und `y_train_wo_outlier` trainiert.

In [ ]:
# To-Do:
# Instantiate the model 


# Fit the model using our train data  with outlier removed and your target variable

# Fit the model using our train data 


#y_pred_test = lin_reg.predict(X_test)
#y_pred_train = lin_reg.predict(X_train)

# Make predictions on the train data and on the test data
y_pred_train_wo_outlier = 
y_pred_test_wo_outlier =

### Modellvergleich

Schauen wir mal, ob sich unsere Bemühungen gelohnt haben:

In [ ]:
from sklearn.metrics import root_mean_squared_error

print('With outliers:')
print("R-squared train: ", round(r2_score(y_train, y_pred_train), 3))
print("R-squared test: ", round(r2_score(y_test, y_pred), 3))
print("RMSE train: ", round(root_mean_squared_error(y_train, y_pred_train), 3))
print("RMSE test: ", round(root_mean_squared_error(y_test, y_pred), 3))
print('-----------------------------------')
print('Without outliers:')
print("R-squared train: ", round(r2_score(y_train_wo_outlier, y_pred_train_wo_outlier), 3))
print("R-squared test: ", round(r2_score(y_test, y_pred_test_wo_outlier), 3))
print("RMSE train: ", round(root_mean_squared_error(y_train_wo_outlier, y_pred_train_wo_outlier), 3))
print("RMSE test: ", round(root_mean_squared_error(y_test, y_pred_test_wo_outlier), 3))


Anscheinend haben wir gute Arbeit geleistet! Das auf den bereinigten Daten trainierte Modell übertrifft das andere Modell in beiden Metriken.

## Abschließende Bemerkung



Gehen Sie mit diesem Notebook vorsichtig um! Wir haben Ihnen gezeigt, wie Sie Ausreißer mit der Z-Score-Methode entfernen können. Aber wir sind nicht auf die geschäftliche Frage eingegangen, ob es sinnvoll ist, alle Ausreißer aus dem Datensatz zu entfernen oder ob es sich um legitime Sonderfälle des Merkmals handelt. Sie möchten so wenig Daten wie möglich verlieren. Denn natürlich, wenn Sie Daten aus dem Datensatz entfernen, wird das Modell auch nicht in der Lage sein, etwas über diese Fälle zu lernen. Sie sollten also vorsichtig sein und sie nur entfernen, wenn es sowohl aus mathematischer/statistischer als auch aus geschäftlicher Sicht sinnvoll ist.

In einigen Fällen können Sie auch Ausreißer aus Ihrer Zielvariable entfernen, aber Sie müssen sicher sein, dass es aus geschäftlicher Perspektive sinnvoll ist, und Sie müssen die Variable in der Produktion verfolgen, um sicher zu sein, dass der Ausreißer nicht häufig auftritt oder sich die Verteilung der Daten ändert. Denn wenn Ihr Modell nicht für alle Kunden funktioniert, könnten Sie diese verlieren.

Hier ist auch ein guter Artikel darüber, wann [To Drop or Not to Drop](https://www.theanalysisfactor.com/outliers-to-drop-or-not-to-drop/)!